# Gaussian Naive Bayes: Pure Python Implementation

In this notebook, we implement the **Gaussian Naive Bayes** classifier from scratch using only pure Python (no scikit-learn).

### Algorithm Overview

1. **Split** the dataset by class label
2. **Summarize** each class: compute the mean and standard deviation for each feature
3. **Calculate probabilities**: for a new data point, compute the Gaussian probability density for each feature given each class
4. **Predict**: choose the class with the highest posterior probability

The Gaussian probability density function:

$$ P(x \mid C) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right) $$

---

## Step 1: Split Data by Class

Group all data rows by their class label. The last column is assumed to be the class label. Returns a dictionary mapping each class to a list of data rows.

In [ ]:
def split_our_data_by_class(dataset):
    splited_data = dict()
    for i in range(len(dataset)):
        vector = dataset[i]
        class_value = vector[-1]
        if (class_value not in splited_data):
            splited_data[class_value] = list()
        splited_data[class_value].append(vector)
    return splited_data

### Create Sample Data

A simple 2-feature, 2-class dataset for demonstration. Each row is `[feature1, feature2, class_label]`.

In [ ]:
dataset = [
    [0.8, 2.3, 0],
    [2.1, 1.6, 0],
    [2.0, 3.6, 0],
    [3.1, 2.5, 0],
    [3.8, 4.7, 0],
    [6.1, 4.4, 1],
    [8.6, 0.3, 1],
    [7.9, 5.3, 1],
    [9.1, 2.5, 1],
    [6.8, 2.7, 1]
]

splited = split_our_data_by_class(dataset)
for label in splited:
    print(f"Class {label}:")
    for row in splited[label]:
        print(f"  {row}")

## Step 2: Calculate Mean and Standard Deviation

For Gaussian NB, we need the **mean** and **standard deviation** of each feature, computed per class. We use the sample standard deviation (dividing by n-1, i.e., Bessel's correction).

In [ ]:
from math import sqrt

def calculate_the_mean(a_list_of_num):
    return sum(a_list_of_num) / float(len(a_list_of_num))

def calculate_the_standard_deviation(a_list_of_num):
    the_mean = calculate_the_mean(a_list_of_num)
    the_variance = sum([(x - the_mean)**2 for x in a_list_of_num]) / float(len(a_list_of_num) - 1)
    return sqrt(the_variance)

### Summarize the Dataset

A `describe` function similar to `pandas.DataFrame.describe()`: computes (mean, std, count) for each feature column. We use `zip(*dataset)` to transpose the data so we iterate over columns instead of rows. The last column (class label) is excluded.

In [ ]:
def describe_our_data(dataset):
    description = [
        (calculate_the_mean(column), calculate_the_standard_deviation(column), len(column))
        for column in zip(*dataset)
    ]
    del description[-1]  # remove the class label column summary
    return description

describe_our_data(dataset)

### Summarize by Class

Compute feature statistics separately for each class. This gives us the \(\mu\) and \(\sigma\) parameters needed for the Gaussian PDF calculation.

In [ ]:
def describe_our_data_by_class(dataset):
    splited_data = split_our_data_by_class(dataset)
    data_description = dict()
    for class_value, rows in splited_data.items():
        data_description[class_value] = describe_our_data(rows)
    return data_description

In [ ]:
description = describe_our_data_by_class(dataset)

print("Full description dict:")
print(description)
print()
print("Class 0 stats (mean, std, count per feature):")
print(description[0])
print()
print("Class 0, Feature 0 stats:")
print(description[0][0])
print()
print("Class 0, Feature 0, count:")
print(description[0][0][2])

[(5.029999999999999, 3.031702858497551, 10), (2.99, 1.5212933094355385, 10)]

## Step 3: Gaussian Probability Density Function

The Gaussian PDF computes the probability density for a given value \(x\), given the mean \(\mu\) and standard deviation \(\sigma\):

$$ f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2} $$

In [ ]:
from math import exp, sqrt, pi

def calculate_the_probability(x, mean, stdev):
    exponent = exp(-((x - mean)**2 / (2 * stdev**2)))
    return (1 / (sqrt(2 * pi) * stdev)) * exponent

# When x equals the mean, we get the peak of the distribution
print(f"P(x=1.0 | mean=1.0, std=1.0) = {calculate_the_probability(1.0, 1.0, 1.0):.6f}")
# When x is 1 std away from mean, density is lower
print(f"P(x=2.0 | mean=1.0, std=1.0) = {calculate_the_probability(2.0, 1.0, 1.0):.6f}")

## Step 4: Calculate Class Probabilities

For a new data point, compute the posterior probability for each class:

$$ P(C \mid X) \propto P(C) \times \prod_{i=1}^{n} P(x_i \mid C) $$

Where:
- \(P(C)\) is the prior: the fraction of training samples belonging to class \(C\)
- \(P(x_i \mid C)\) is computed using the Gaussian PDF with the class-specific mean and std for feature \(i\)

In [ ]:
def calculate_class_probability(description, row):
    # Total training samples across all classes
    total_rows = sum([description[label][0][2] for label in description])
    probabilities = dict()
    for class_value, class_description in description.items():
        # Start with the prior P(class)
        probabilities[class_value] = description[class_value][0][2] / float(total_rows)
        # Multiply by each feature's Gaussian likelihood
        for i in range(len(class_description)):
            mean, stdev, count = class_description[i]
            probabilities[class_value] *= calculate_the_probability(row[i], mean, stdev)
    return probabilities

## Summary

We have built a complete Gaussian Naive Bayes classifier from scratch:

| Step | Function | Purpose |
|------|----------|--------|
| 1 | `split_our_data_by_class` | Group data by class label |
| 2 | `describe_our_data_by_class` | Compute per-class feature statistics (mean, std) |
| 3 | `calculate_the_probability` | Gaussian PDF for a single feature value |
| 4 | `calculate_class_probability` | Combine prior and likelihoods for prediction |

To make a prediction, we call `calculate_class_probability` and select the class with the highest probability.